# Train a hybrid ModernBERT safety router

This is the recommended, end-to-end proof-of-concept notebook. It trains a rank-4 LoRA adapter on ModernBERT to predict **whether each candidate preserves the strongest model's quality**. Analytical latency then chooses the fastest candidate predicted safe. Candidate LLMs are never loaded or timed here.

You will:

1. inspect a pre-collected LLMRouterBench release;
2. describe each candidate using model-card facts;
3. calculate latency from model size, architecture, precision, and prompt size;
4. build hindsight-oracle labels from pre-collected quality;
5. train ModernBERT with safety BCE plus an auxiliary oracle loss;
6. freeze the validation policy before opening the test split; and
7. export reports plus a reconstructable LoRA artifact.

> The result proves routing feasibility under stated analytical assumptions. It does not claim measured production latency.

## 1. One-cell Google Colab setup

Open this notebook in Google Colab, select a GPU runtime, and run every cell in order. This cell clones the `develop` branch, installs the project normally (not as an editable package), registers `src` in the live kernel, and verifies the import immediately. No terminal, runtime restart, or separate setup notebook is required.

In [ ]:
%cd /content
!test -d /content/LLM_Router || git clone --branch develop https://github.com/BrunoVitti96/LLM-router.git /content/LLM_Router
!git -C /content/LLM_Router pull --ff-only origin develop
%cd /content/LLM_Router
%pip install -q -U ".[notebook]"

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LLM_Router")
SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
import llm_router

print(f"Router package ready from {Path(llm_router.__file__).resolve()}")

## 2. Imports, reproducibility, and automatic benchmark download

The official pre-collected LLMRouterBench archive is about 1.28 GB. The cell downloads it once from Hugging Face and extracts it under `/content`; it does not install or execute any candidate LLM. Outputs stay in the router repository, separate from the immutable benchmark evidence.

In [ ]:
import shutil
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display

from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    benchmark_inventory,
    export_public_benchmark,
    load_llmrouterbench,
    make_complete_panel,
    run_public_benchmark,
    simulate_economics,
    split_benchmark,
)
from llm_router.utils.training import seed_everything

SEED = 42
DATA_ROOT = Path("/content/LLMRouterBench")
OUTPUT_DIR = Path("reports_benchmark/modernbert_hybrid_notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

seed_everything(SEED)
bench_candidates = list(DATA_ROOT.rglob("bench")) if DATA_ROOT.exists() else []
if not bench_candidates:
    archive = hf_hub_download(
        repo_id="NPULH/LLMRouterBench",
        filename="bench-release.tar.gz",
        repo_type="dataset",
    )
    results_dir = DATA_ROOT / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as bundle:
        bundle.extractall(results_dir, filter="data")
    bench_candidates = list(DATA_ROOT.rglob("bench"))
assert bench_candidates, f"Benchmark extraction failed under {DATA_ROOT}"
DATA_ROOT = bench_candidates[0]
print({"device": DEVICE, "data_root": str(DATA_ROOT.resolve())})
if DEVICE == "cpu":
    print("Warning: CPU training works for a smoke test but will be slow. A GPU is recommended.")

## 3. Inspect available datasets and models

Do this before editing model profiles. The strings in `SELECTED_MODELS` must exactly match model directory names shown below. Choose at least two candidates and at least three datasets for a dataset-disjoint split.

In [ ]:
inventory = benchmark_inventory(DATA_ROOT)
assert not inventory.empty, "No LLMRouterBench result files were found."
inventory_summary = (
    inventory.groupby(["model", "dataset", "source_split"])
    .size()
    .rename("files")
    .reset_index()
)
display(inventory_summary)
print(f"Available models: {inventory.model.nunique()}")
print(f"Available datasets: {inventory.dataset.nunique()}")

## 4. Declare the candidate model profiles

The defaults below are an immediately runnable three-model panel from LLMRouterBench: a 7B, an 8B, and a 9B autoregressive model. The names and approximate parameter classes come from the [official benchmark model pool](https://github.com/ynulihao/LLMRouterBench#model-pools). They create latency variation without running candidate inference.

You may customize this dictionary using exact names from the inventory above. Parameter counts, active parameter counts, architecture, and precision should come from model cards or configuration files. A diffusion candidate needs pre-collected quality results in the benchmark plus its denoising steps and generated block size; the official default pool contains no diffusion model. Prices are zero because this notebook optimizes latency only. Hardware and overhead values remain explicit POC assumptions—not measurements.

In [ ]:
MODEL_PROFILES = {
    "Fin-R1": {
        "parameters_billions": 7.0,
        "active_parameters_billions": 7.0,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
    "Qwen3-8B": {
        "parameters_billions": 8.0,
        "active_parameters_billions": 8.0,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
    "NVIDIA-Nemotron-Nano-9B-v2": {
        "parameters_billions": 9.0,
        "active_parameters_billions": 9.0,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
}

SELECTED_MODELS = tuple(MODEL_PROFILES)
assert len(SELECTED_MODELS) >= 2, "Routing requires at least two models."
missing_models = sorted(set(SELECTED_MODELS) - set(inventory.model))
assert not missing_models, (
    f"Configured models were not found: {missing_models}. "
    f"Available models: {sorted(inventory.model.unique())}"
)
print(SELECTED_MODELS)

## 5. Load pre-collected quality and calculate analytical latency

The benchmark provides candidate answers and quality scores. `simulate_economics` calculates latency without loading those candidates. Expected output length is a bounded function of prompt length, so the policy cannot peek at a candidate's realized response length.

In [ ]:
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        **settings,
    )
    for name, settings in MODEL_PROFILES.items()
)
scenario = EconomicsScenario(
    name="notebook-analytical-latency-poc",
    as_of=pd.Timestamp.utcnow().date().isoformat(),
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="POC assumptions; not measured production latency.",
)

records = load_llmrouterbench(DATA_ROOT, models=SELECTED_MODELS)
simulated = simulate_economics(records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)
assert simulated.latency_source.eq("analytical").all()
print({"complete_prompts": len(panel.examples), "models": panel.models})
display(
    simulated.groupby("model")["simulated_latency_s"]
    .agg(["min", "median", "mean", "max"])
    .sort_values("mean")
)

### Leakage check

This deliberately changes every realized completion length. Analytical latency must remain identical because only prompt size and declared model/scenario properties are allowed to affect it.

In [ ]:
counterfactual_records = records.copy()
counterfactual_records["completion_tokens"] *= 100
counterfactual = simulate_economics(counterfactual_records, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual.simulated_latency_s,
), "Analytical latency unexpectedly depends on realized completion length."
print("Passed: candidate completion tokens do not affect analytical latency.")

## 6. Freeze train, validation, and sealed-test splits

`dataset_ood` is the stronger POC: entire datasets are held out, so ModernBERT must generalize beyond training dataset identities. Use `random` only as an easier interpolation diagnostic. The fallback is selected from **training quality only**.

In [ ]:
split = split_benchmark(panel, mode="dataset_ood", seed=SEED)
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "prompts": [len(split.train), len(split.validation), len(split.test)],
        "datasets": [
            split.train_datasets,
            split.validation_datasets,
            split.test_datasets,
        ],
    }
)
display(split_summary)
assert set(split.train_datasets).isdisjoint(split.validation_datasets)
assert set(split.train_datasets).isdisjoint(split.test_datasets)
assert set(split.validation_datasets).isdisjoint(split.test_datasets)

## 7. Inspect validation oracle headroom before training

For each prompt, the oracle chooses the analytically fastest model whose observed score is no worse than the training-selected fallback. It is an unattainable upper bound because it uses answer outcomes. This diagnostic uses validation only—the sealed test remains unopened until the policy is frozen. If validation shows little oracle headroom, no prompt-only router can prove much under this candidate panel and scenario.

In [ ]:
fallback_index = int(panel.score[split.train].mean(axis=0).argmax())
fallback_model = panel.models[fallback_index]
oracle_target = oracle_choices(
    panel.score, panel.latency, fallback_index=fallback_index
)
validation_rows = np.arange(len(split.validation))
validation_oracle = oracle_target[split.validation]
oracle_quality = panel.score[split.validation][validation_rows, validation_oracle].mean()
fallback_quality = panel.score[split.validation, fallback_index].mean()
oracle_latency = panel.latency[split.validation][validation_rows, validation_oracle].mean()
fallback_latency = panel.latency[split.validation, fallback_index].mean()

oracle_summary = pd.Series(
    {
        "fallback_model": fallback_model,
        "fallback_quality": fallback_quality,
        "oracle_quality": oracle_quality,
        "oracle_quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "fallback_latency_s": fallback_latency,
        "oracle_latency_s": oracle_latency,
        "oracle_latency_savings": 1 - oracle_latency / fallback_latency,
        "oracle_fallback_usage": np.mean(validation_oracle == fallback_index),
    },
    name="validation oracle upper bound",
)
display(oracle_summary.to_frame())
display(
    pd.Series(np.array(panel.models)[oracle_target[split.train]])
    .value_counts(normalize=True)
    .rename("train_oracle_rate")
)

## 8. Understand the objective and training loss

For each non-fallback candidate, the deployable target is replacement safety:

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_f(x)-\epsilon_q].$$

ModernBERT's safety head emits independent logits $s_m$ and learns them with binary cross-entropy:

$$\mathcal L_{safety}=\frac{1}{N(M-1)}\sum_{x,m}BCEWithLogits(s_m(x),y_m(x)).$$

A second, training-only head imitates the hindsight oracle. Its loss combines oracle-class cross-entropy, expected quality risk, and latency regret:

$$\mathcal L_{oracle}=(1+g_o)CE(z,o)+4\sum_m p_m d_m+\sum_m p_m r_m.$$

The complete training loss is:

$$\boxed{\mathcal L_{train}=1.0\,\mathcal L_{safety}+0.25\,\mathcal L_{oracle}}.$$

The oracle head shapes the shared representation but is never used for deployment. At route time, analytical latency solves the actual per-prompt objective:

$$\pi(x)=\arg\min_m \widehat L_m(x)\quad\text{subject to}\quad \widehat P_m(safe\mid x)\ge\tau,$$

with the fallback always eligible and alternatives required to be at least 2% faster. Finally, the whole router is disabled unless validation has positive net analytical savings and a one-sided 95% quality-retention lower bound of at least 98%.

## 9. Train ModernBERT

This downloads the pinned `nomic-ai/modernbert-embed-base`, freezes its base weights, inserts rank-4 LoRA adapters, and trains the LoRA parameters plus safety and auxiliary-oracle heads. Five epochs are appropriate for the first POC; increase only after inspecting validation loss. Candidate LLMs are not loaded.

In [ ]:
training = train_modernbert_hybrid_poc(
    panel,
    split,
    epochs=5,
    batch_size=8,
    learning_rate=1e-4,
    quality_epsilon=0.0,
    safety_loss_weight=1.0,
    oracle_auxiliary_weight=0.25,
    device=DEVICE,
)
display(training.history)
print({"training_seconds": round(training.training_seconds, 1)})
assert training.safety_probabilities.shape == panel.score.shape
assert np.allclose(
    training.safety_probabilities[:, training.fallback_index], 1.0
)
assert np.all(
    (training.safety_probabilities >= 0)
    & (training.safety_probabilities <= 1)
)

## 10. Select the validation policy, then open the sealed test

The selector searches confidence thresholds using validation only. A proposed alternative must be at least 2% faster analytically. The router activates only when validation has positive net savings after assumed router overhead and its one-sided 95% quality-retention lower bound is at least 98%. Test quality is consumed only after that policy is frozen.

In [ ]:
result = run_public_benchmark(
    panel,
    split,
    objective="latency",
    minimum_quality_retention=0.98,
    confidence=0.95,
    minimum_predicted_savings=0.02,
    router_overhead_s=scenario.router_overhead_s,
    seed=SEED,
    routing_probabilities=training.safety_probabilities,
    router_name="modernbert_hybrid_router",
)
assert result.fallback_model == fallback_model
display(result.threshold_search)
display(result.summary)
print(
    {
        "router_active": result.router_active,
        "selected_threshold": result.selected_threshold,
        "fallback_model": result.fallback_model,
    }
)

### Interpret the result

A convincing POC has oracle headroom, non-trivial ModernBERT alternative usage, at least 98% quality retention at the one-sided 95% lower bound, positive analytical savings after router overhead, and similar conclusions under reasonable scenario changes.

If `router_active` is false, that is a valid result. Compare the `outcome_oracle` and `modernbert_hybrid_router` rows:

- weak oracle savings means the candidate panel or assumptions offer little headroom;
- strong oracle savings but fallback-only routing points to prediction/generalization difficulty;
- good random-split results but weak dataset-OOD results indicate dataset-specific overfitting;
- good quality but weak savings suggests the confidence threshold or router-overhead assumption is too costly.

## 11. Export reproducible reports and the router artifact

The report manifest records every analytical assumption. The router artifact contains the LoRA adapter, routing head, tokenizer, base encoder revision, selected confidence threshold, fallback, and deployment guard state.

In [ ]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
artifact_dir = export_modernbert_hybrid_poc(
    training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    minimum_predicted_savings=0.02,
)
print({"reports": str(report_dir.resolve()), "artifact": str(artifact_dir.resolve())})

## 12. Download the trained artifact from Colab

Colab storage is temporary. This cell packages the reports and trained router, then opens a browser download so the result survives runtime shutdown.

In [ ]:
bundle_path = shutil.make_archive(
    str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR
)
print(f"Created {bundle_path}")
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    print("Not running in Colab; download the ZIP from the printed path.")

## 13. Final POC checklist

Before presenting the result, confirm:

- the default model profiles ran unchanged, or every customized profile is sourced;
- no candidate inference was run to construct latency;
- the completion-length leakage assertion passed;
- fallback selection used training quality only;
- confidence and activation were selected on validation only;
- the test split was opened once after policy freeze;
- results are described as analytical or simulated latency, never measured latency; and
- the conclusion survives multiple reasonable hardware and diffusion-step scenarios.